<a href="https://colab.research.google.com/github/Anil-Pradhan-web/touch-grass-nature-advisor/blob/main/Phantom_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%bash
pip install -q gradio transformers torch accelerate

In [ ]:
import gradio as gr
import torch
from transformers import pipeline

# 1. Initialize the lightweight open-weight Hugging Face model
print("Summoning the spirits (loading Flan-T5 model)... This may take a moment.")
# We use flan-t5-large as it fits comfortably in Colab's free-tier RAM and generates good structured outputs
oracle_pipeline = pipeline("text2text-generation", model="google/flan-t5-large", device=0 if torch.cuda.is_available() else -1)
print("The spirits have arrived! PhantomTerminal AI is ready.")

# 2. Define the core oracle logic
def consult_oracle(code_input, persona):
    if not code_input.strip():
        return "The terminal is dead silent... Type something to wake the spirits.", "Ensure you provide a command or error log.", "Low"

    # Customize prompts based on the chosen spirit persona
    if persona == "Ancient FORTRAN Ghost":
        prompt_lore = f"Rewrite this code/error as an ancient spooky curse or ancient machine folklore: '{code_input}'"
        prompt_fix = f"Explain how to fix this programming issue like a mystical exorcism ritual: '{code_input}'"
    elif persona == "Sarcastic Haunted Terminal":
        prompt_lore = f"Mock this code/error sarcastically as a snarky ghost trapped in a CRT screen: '{code_input}'"
        prompt_fix = f"Give practical debugging advice for this, but make it sound incredibly condescending and ghostly: '{code_input}'"
    else:  # Cyberpunk Poltergeist
        prompt_lore = f"Describe this code/error as a neon glitch, a digital entity haunting the mainframe: '{code_input}'"
        prompt_fix = f"Give a highly technical cyberpunk debugging patch styled as purging a malware daemon: '{code_input}'"

    # Generate Supernatural Commentary
    lore_output = oracle_pipeline(prompt_lore, max_length=150, num_return_sequences=1)[0]['generated_text']

    # Generate Ghostly Fix
    fix_output = oracle_pipeline(prompt_fix, max_length=150, num_return_sequences=1)[0]['generated_text']

    # Determine Haunted Threat Level based on input length or keywords
    threat_keywords = ["error", "fail", "exception", "segmentation fault", "null", "dead"]
    score = sum(1 for word in threat_keywords if word in code_input.lower())

    if score == 0:
        threat_level = "Low (Quiet Whispers)"
    elif score <= 2:
        threat_level = "Spooky (Ectoplasmic Activity Detected)"
    else:
        threat_level = "Fully Possessed (Call an Exorcist / Senior Developer)"

    return lore_output, fix_output, threat_level

# 3. Build the custom spooky Gradio Interface
css = """
body { background-color: #0d0208; color: #00ff41; font-family: 'Courier New', Courier, monospace; }
.gradio-container { background-color: #0a0512 !important; border: 2px solid #ff0055; }
input, textarea, select { background-color: #1a0f2b !important; color: #ff0055 !important; border: 1px solid #00ff41 !important; }
button { background-color: #ff0055 !important; color: white !important; font-weight: bold; }
"""

with gr.Blocks(theme=gr.themes.Monochrome(), css=css) as app:
    gr.Markdown(
        """
        # 🎃 PhantomTerminal AI 💀
        ### *The Haunted Code & Terminal Oracle (Hacktoberfest Week 4 - Ghost in the Machine)*
        *Paste your cursed code, broken terminal commands, or terrifying error logs and let the machine's spirits guide you.*
        """
    )

    with gr.Row():
        with gr.Column():
            code_input = gr.TextArea(
                label="Cursed Code, Terminal Command, or Error Log",
                placeholder="paste your segmentation fault, traceback, or buggy code here...",
                lines=8
            )
            persona_dropdown = gr.Dropdown(
                choices=["Ancient FORTRAN Ghost", "Sarcastic Haunted Terminal", "Cyberpunk Poltergeist"],
                value="Ancient FORTRAN Ghost",
                label="Select Spirit Persona"
            )
            submit_btn = gr.Button("Perform Exorcism (Analyze Code)")

        with gr.Column():
            commentary_output = gr.Textbox(label="🔮 Eerie Supernatural Commentary", interactive=False)
            fix_output = gr.Textbox(label="🕯️ Ghostly Fix / Advice", interactive=False)
            threat_output = gr.Textbox(label="🚨 Haunted Threat Level", interactive=False)

    submit_btn.click(
        fn=consult_oracle,
        inputs=[code_input, persona_dropdown],
        outputs=[commentary_output, fix_output, threat_output]
    )

# 4. Launch the app with a public shareable link
app.launch(share=True)